# HPO Tables And Early-Stopping Tables

This notebook is an exploratory helper for creating thesis appendix tables from existing MLflow runs. It does not run new experiments. Update the run IDs in the configuration cell, then execute the sections you need.

Outputs are written below `src/visuals/raw/` by default.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.Orchestration.helpers import VISUALS, fetch_direct_child_runs
from src.Orchestration.ablation_metrics import summarize_bde_early_stopping_epochs

RAW_OUT = VISUALS / "raw"
RAW_OUT.mkdir(parents=True, exist_ok=True)

ROOT, RAW_OUT

/Users/kingmopser/BachelorThesis/BachelorsThesisCode/.pixi/envs/default/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(PosixPath('/Users/kingmopser/BachelorThesis/BachelorsThesisCode'),
 PosixPath('/Users/kingmopser/BachelorThesis/BachelorsThesisCode/src/visuals/raw'))

## Configuration

Set the MLflow experiment/run IDs for the HPO run you want to summarize. The default values below correspond to the existing Miami Housing HPO run used during thesis analysis.

In [2]:
DATASET_SLUG = "miami_housing"
EXPERIMENT_ID = "2"
HPO_PARENT_RUN_ID = "7e99f7fdc8a44f1b9461c3c1bdc6ec46"

RAW_TABLE_PATH = RAW_OUT / f"hpo_raw_{DATASET_SLUG}.tex"
AGG_WINKLER_PATH = RAW_OUT / f"hpo_{DATASET_SLUG}_agg_winkler.tex"
AGG_NLL_PATH = RAW_OUT / f"hpo_{DATASET_SLUG}_agg_nll.tex"
COVERAGE_PATH = RAW_OUT / f"{DATASET_SLUG}_HPO_coverage.tex"

## Load HPO Child Runs

This fetches the direct child runs of one HPO parent run. Each child run corresponds to one HPO trial.

In [3]:
df, exp_name = fetch_direct_child_runs(
    experiment_id=EXPERIMENT_ID,
    run_id=HPO_PARENT_RUN_ID,
)

print(exp_name)
df.head()

Experiment_Dataset_miami_housing


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Negative_log_likelihood,metrics.Winkler_Coverage,metrics.Mean_Winkler_Score,metrics.RMSE,...,tags.mlflow.source.git.commit,tags.mlflow.parentRunId,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.source.name,experiment_name,parent_run_id,parent_run_name,parent_experiment_id,parent_experiment_name
0,87eca9b70f7c412eb60676c84c740057,2,FINISHED,file:///Users/kingmopser/BachelorThesis/Bachel...,2026-05-21 00:25:02.838000+00:00,2026-05-21 00:25:30.225000+00:00,-0.596145,0.912432,0.909036,0.328707,...,cc2565d3909e27a8fa9952b80bdc9f924d1a41b8,7e99f7fdc8a44f1b9461c3c1bdc6ec46,kingmopser,LOCAL,main.py,Experiment_Dataset_miami_housing,7e99f7fdc8a44f1b9461c3c1bdc6ec46,HPO_Study_miami_housing,2,Experiment_Dataset_miami_housing
1,e3241625a0e040f699247f450c212617,2,FINISHED,file:///Users/kingmopser/BachelorThesis/Bachel...,2026-05-21 00:21:39.339000+00:00,2026-05-21 00:25:02.830000+00:00,-0.661759,0.911978,0.844207,0.311941,...,cc2565d3909e27a8fa9952b80bdc9f924d1a41b8,7e99f7fdc8a44f1b9461c3c1bdc6ec46,kingmopser,LOCAL,main.py,Experiment_Dataset_miami_housing,7e99f7fdc8a44f1b9461c3c1bdc6ec46,HPO_Study_miami_housing,2,Experiment_Dataset_miami_housing
2,77cac1410e0a4d85b0c3a64cc92c494c,2,FINISHED,file:///Users/kingmopser/BachelorThesis/Bachel...,2026-05-21 00:20:19.039000+00:00,2026-05-21 00:21:39.333000+00:00,-0.636511,0.884755,0.843236,0.309763,...,cc2565d3909e27a8fa9952b80bdc9f924d1a41b8,7e99f7fdc8a44f1b9461c3c1bdc6ec46,kingmopser,LOCAL,main.py,Experiment_Dataset_miami_housing,7e99f7fdc8a44f1b9461c3c1bdc6ec46,HPO_Study_miami_housing,2,Experiment_Dataset_miami_housing
3,db76dbb9757e4afcb6640913addb1bac,2,FINISHED,file:///Users/kingmopser/BachelorThesis/Bachel...,2026-05-21 00:20:00.658000+00:00,2026-05-21 00:20:19.033000+00:00,-0.622101,0.915608,0.895944,0.323370,...,cc2565d3909e27a8fa9952b80bdc9f924d1a41b8,7e99f7fdc8a44f1b9461c3c1bdc6ec46,kingmopser,LOCAL,main.py,Experiment_Dataset_miami_housing,7e99f7fdc8a44f1b9461c3c1bdc6ec46,HPO_Study_miami_housing,2,Experiment_Dataset_miami_housing
4,3d21a6e364424aa4b1c3674287eb1a94,2,FINISHED,file:///Users/kingmopser/BachelorThesis/Bachel...,2026-05-21 00:18:06.360000+00:00,2026-05-21 00:20:00.651000+00:00,-0.673819,0.920145,0.841651,0.311858,...,cc2565d3909e27a8fa9952b80bdc9f924d1a41b8,7e99f7fdc8a44f1b9461c3c1bdc6ec46,kingmopser,LOCAL,main.py,Experiment_Dataset_miami_housing,7e99f7fdc8a44f1b9461c3c1bdc6ec46,HPO_Study_miami_housing,2,Experiment_Dataset_miami_housing


## Raw HPO Table

Exports one row per HPO trial with the selected parameters and metrics.

In [4]:
cols = [
    "metrics.RMSE",
    "metrics.Mean_Winkler_Score",
    "metrics.Negative_log_likelihood",
    "metrics.Winkler_Coverage",
    "params.hidden_layers",
    "params.desired_energy_var_end",
    "params.n_samples",
]

available_cols = [col for col in cols if col in df.columns]
table_df = df.loc[:, available_cols].copy()

table_df = table_df.rename(
    columns={
        "metrics.RMSE": "RMSE",
        "metrics.Mean_Winkler_Score": "WinklerScore",
        "metrics.Negative_log_likelihood": "NLL",
        "metrics.Winkler_Coverage": "WinklerCoverage",
        "params.hidden_layers": "HiddenLayers",
        "params.desired_energy_var_end": "Energy_Var_End",
        "params.n_samples": "N_Samples",
    }
)

for col in ["RMSE", "WinklerScore", "NLL", "WinklerCoverage"]:
    if col in table_df.columns:
        table_df[col] = pd.to_numeric(table_df[col], errors="coerce").map(lambda x: f"{x:.4f}")

if "HiddenLayers" in table_df.columns:
    table_df = table_df.sort_values(["HiddenLayers"])

RAW_TABLE_PATH.write_text(table_df.to_latex(index=False, escape=False))
RAW_TABLE_PATH

PosixPath('/Users/kingmopser/BachelorThesis/BachelorsThesisCode/src/visuals/raw/hpo_raw_miami_housing.tex')

## Aggregated HPO Tables

Groups HPO trials into compact appendix tables. The grouping functions are intentionally simple and can be adjusted per experiment.

In [5]:
def classify_energy_variance(value):
    value = str(value)
    if value in {"0.1", "0.01"}:
        return "[0.1, 0.01]"
    return "[0.001, 0.0001]"


def classify_samples(value):
    value = str(value)
    if value in {"1000", "5000"}:
        return "[1000, 5000]"
    return "[200, 500]"

work = df.copy()
work["energy_var_group"] = work["params.desired_energy_var_end"].apply(classify_energy_variance)
work["sample_group"] = work["params.n_samples"].apply(classify_samples)

metric_cols = [
    "metrics.Mean_Winkler_Score",
    "metrics.Negative_log_likelihood",
    "metrics.Winkler_Coverage",
]

appendix_table = (
    work.groupby(["energy_var_group", "sample_group", "params.hidden_layers"])[metric_cols]
    .agg(["min", "max", "mean", "std"])
    .round(3)
)

rename_metrics = {
    "metrics.Mean_Winkler_Score": "winkler",
    "metrics.Negative_log_likelihood": "nll",
    "metrics.Winkler_Coverage": "coverage",
}

appendix_table.columns = [
    f"{rename_metrics[metric]}_{stat}"
    for metric, stat in appendix_table.columns
]

appendix_table = appendix_table.reset_index().rename(
    columns={
        "energy_var_group": "Energy Var.",
        "sample_group": "Samples",
        "params.hidden_layers": "Layers",
    }
)
appendix_table = appendix_table.sort_values(["Energy Var.", "Samples", "Layers"])

base_cols = ["Energy Var.", "Samples", "Layers"]

winkler_table = appendix_table[
    base_cols + ["winkler_min", "winkler_max", "winkler_mean", "winkler_std"]
].rename(columns={
    "winkler_min": "Min",
    "winkler_max": "Max",
    "winkler_mean": "Mean",
    "winkler_std": "SD",
})

nll_table = appendix_table[
    base_cols + ["nll_min", "nll_max", "nll_mean", "nll_std"]
].rename(columns={
    "nll_min": "Min",
    "nll_max": "Max",
    "nll_mean": "Mean",
    "nll_std": "SD",
})

coverage_table = appendix_table[
    base_cols + ["coverage_min", "coverage_max", "coverage_mean", "coverage_std"]
].rename(columns={
    "coverage_min": "Min",
    "coverage_max": "Max",
    "coverage_mean": "Mean",
    "coverage_std": "SD",
})

AGG_WINKLER_PATH.write_text(winkler_table.to_latex(index=False, escape=False))
AGG_NLL_PATH.write_text(nll_table.to_latex(index=False, escape=False))
COVERAGE_PATH.write_text(coverage_table.to_latex(index=False, escape=False))

AGG_WINKLER_PATH, AGG_NLL_PATH, COVERAGE_PATH

(PosixPath('/Users/kingmopser/BachelorThesis/BachelorsThesisCode/src/visuals/raw/hpo_miami_housing_agg_winkler.tex'),
 PosixPath('/Users/kingmopser/BachelorThesis/BachelorsThesisCode/src/visuals/raw/hpo_miami_housing_agg_nll.tex'),
 PosixPath('/Users/kingmopser/BachelorThesis/BachelorsThesisCode/src/visuals/raw/miami_housing_HPO_coverage.tex'))

In [ ]:
appendix_table

## Early-Stopping Epoch Table

This section summarizes BDE early-stopping lengths from selected robustness runs. Update the run IDs if the robustness runs change.

In [6]:
EARLY_STOPPING_RUNS = {
    "miami_housing": {"experiment_id": "2", "robustness_run_id": "c45bebd028884dbc86693af5a9168ad1"},
    "healthcare_insurance": {"experiment_id": "3", "robustness_run_id": "7b2af92870ec4baba808d191a3cd2a0e"},
    "wine_quality": {"experiment_id": "4", "robustness_run_id": "cc6535cff1bf4d4db04ec7674975252a"},
    "fiat500": {"experiment_id": "5", "robustness_run_id": "2238dc052ff84c95b7b6781cd3f7313a"},
}

early_stopping_results = {
    dataset: summarize_bde_early_stopping_epochs(**run_spec)
    for dataset, run_spec in EARLY_STOPPING_RUNS.items()
}

early_stopping_results

{'miami_housing': {'experiment_id': '2',
  'robustness_run_id': 'c45bebd028884dbc86693af5a9168ad1',
  'robustness_run_name': 'Robustnesstest_Data_miami_housing',
  'bde_run_id': 'efc61d350eca42b1a0694d5be0aa9b26',
  'seed_epochs': {'seed0': 478, 'seed1': 501, 'seed2': 552},
  'mean_epoch': 510.3333333333333,
  'std_epoch': 37.872593432894625,
  'n_seeds': 3,
  'summary': '510.3 \\pm 37.9'},
 'healthcare_insurance': {'experiment_id': '3',
  'robustness_run_id': '7b2af92870ec4baba808d191a3cd2a0e',
  'robustness_run_name': 'Robustnesstest_Data_healthcare_insurance_expenses',
  'bde_run_id': '4c25a290659741d7aa4f508db4c11169',
  'seed_epochs': {'seed0': 259, 'seed1': 223, 'seed2': 259},
  'mean_epoch': 247,
  'std_epoch': 20.784609690826528,
  'n_seeds': 3,
  'summary': '247.0 \\pm 20.8'},
 'wine_quality': {'experiment_id': '4',
  'robustness_run_id': 'cc6535cff1bf4d4db04ec7674975252a',
  'robustness_run_name': 'Robustnesstest_Data_wine_quality',
  'bde_run_id': '5323d2c78e284c94afd1d7b3ee

In [7]:
latex = []
latex.append(r"\begin{tabular}{l r}")
latex.append(r"\toprule")
latex.append(r"Dataset & Mean early stopping epoch ")
latex.append(r"\midrule")

for dataset, result in early_stopping_results.items():
    dataset_name = dataset.replace("_", r"\_")
    latex.append(
        f"{dataset_name} & {result['mean_epoch']:.1f} $\pm$ {result['std_epoch']:.1f} \\"
    )

latex.append(r"\bottomrule")
latex.append(r"\end{tabular}")

latex_table = "".join(latex)
early_stopping_path = RAW_OUT / "early_stopping_table.tex"
early_stopping_path.write_text(latex_table)
early_stopping_path

PosixPath('/Users/kingmopser/BachelorThesis/BachelorsThesisCode/src/visuals/raw/early_stopping_table.tex')